# Занятие 5. Связи таблиц и FOREIGN KEY

## Цель занятия

На прошлом занятии мы научились проектировать таблицы и выбирать типы данных.

Сегодня изучаем очень важную тему: **связи между таблицами**.

К концу занятия студент должен понимать:

- зачем данные делят на несколько таблиц;
- что такое связь `один ко многим`;
- что такое `FOREIGN KEY`;
- как связать две таблицы;
- как включить поддержку внешних ключей в SQLite;
- почему связи таблиц нужны в дипломном проекте.

---

## Зачем нужны связи таблиц

В реальном проекте данные редко хранятся в одной большой таблице.

Например, в интернет-магазине есть:

- пользователи;
- заказы;
- товары.

Если хранить всё в одной таблице, данные будут повторяться.

Пример плохой структуры:

| user_name | email | order_title | order_price |
|---|---|---|---|
| Анна | anna@mail.ru | Ноутбук | 70000 |
| Анна | anna@mail.ru | Мышь | 1500 |

Имя и email Анны повторяются.

Лучше сделать две таблицы:

`users`:

| id | name | email |
|---|---|---|
| 1 | Анна | anna@mail.ru |

`orders`:

| id | user_id | title | price |
|---|---|---|---|
| 1 | 1 | Ноутбук | 70000 |
| 2 | 1 | Мышь | 1500 |

Теперь данные пользователя хранятся один раз.

---

## Что такое связь один ко многим

Связь **один ко многим** означает:

одна запись в первой таблице может быть связана со многими записями во второй таблице.

Примеры:

- один пользователь → много заказов;
- один кандидат → много собеседований;
- один студент → много оценок;
- один пользователь → много тренировок;
- один приём пищи → много продуктов.

---

## Что такое FOREIGN KEY

`FOREIGN KEY` — это внешний ключ.

Он связывает одну таблицу с другой.

Пример:

```sql
user_id INTEGER,
FOREIGN KEY (user_id) REFERENCES users(id)
```

Это означает:

- в таблице `orders` есть поле `user_id`;
- оно ссылается на поле `id` таблицы `users`;
- каждый заказ принадлежит конкретному пользователю.

---

## Важно про SQLite

В SQLite поддержку внешних ключей нужно включать отдельно:

```python
cursor.execute("PRAGMA foreign_keys = ON")
```

Без этого SQLite может не проверять связи строго.

---

## Зачем это нужно в дипломе

В дипломном проекте связи таблиц нужны почти всегда.

Примеры:

HR:
- кандидат → собеседования;
- вакансия → кандидаты.

Питание:
- пользователь → приёмы пищи;
- приём пищи → продукты.

Спорт:
- пользователь → тренировки;
- тренировка → упражнения.

Магазин:
- клиент → заказы;
- заказ → товары.

---

## Связь с GitHub

На этом занятии студент должен сохранить:

- solved-ноутбук;
- TODO-ноутбук;
- SQL-код создания связанных таблиц;
- пример своей связи таблиц для диплома.

Рекомендуемая ветка:

```bash
git checkout main
git pull origin main
git checkout -b feature/lesson-05-relations
```

Рекомендуемый commit:

```bash
git add .
git commit -m "feat: add foreign keys and table relations"
git push -u origin feature/lesson-05-relations
```

## Ячейка 1. Подключаем sqlite3 и создаём базу

Создаём базу `relations_lesson05.db`.

В ней будут две связанные таблицы:

- `users`;
- `orders`.

In [1]:
import sqlite3

connection = sqlite3.connect("relations_lesson05.db")
cursor = connection.cursor()

print("База данных подключена")

База данных подключена


## Ячейка 2. Включаем поддержку FOREIGN KEY

В SQLite внешние ключи нужно включать отдельно.

In [2]:
cursor.execute("PRAGMA foreign_keys = ON")

print("Поддержка FOREIGN KEY включена")

Поддержка FOREIGN KEY включена


## Ячейка 3. Создаём таблицу users

Это главная таблица.

У пользователя может быть много заказов.

In [3]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    email TEXT
)
""")

connection.commit()

print("Таблица users создана")

Таблица users создана


## Ячейка 4. Создаём таблицу orders

Таблица `orders` содержит поле `user_id`.

Это внешний ключ, который ссылается на `users(id)`.

In [4]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER,
    title TEXT,
    price INTEGER,
    FOREIGN KEY (user_id) REFERENCES users(id)
)
""")

connection.commit()

print("Таблица orders создана")

Таблица orders создана


## Ячейка 5. Очищаем таблицы перед новым запуском

Сначала очищаем дочернюю таблицу `orders`, потом родительскую `users`.

Это важно, потому что заказы зависят от пользователей.

In [5]:
cursor.execute("DELETE FROM orders")
cursor.execute("DELETE FROM users")

connection.commit()

print("Таблицы очищены")

Таблицы очищены


## Ячейка 6. Добавляем пользователей

Добавим двух пользователей.

In [6]:
users = [
    ("Анна", "anna@example.com"),
    ("Иван", "ivan@example.com")
]

cursor.executemany(
    "INSERT INTO users (name, email) VALUES (?, ?)",
    users
)

connection.commit()

print("Пользователи добавлены")

Пользователи добавлены


## Ячейка 7. Получаем id пользователей

Чтобы добавить заказ, нужно знать `id` пользователя.

In [7]:
cursor.execute("SELECT * FROM users")

users_data = cursor.fetchall()

for user in users_data:
    print(user)

anna_id = users_data[0][0]
ivan_id = users_data[1][0]

print("ID Анны:", anna_id)
print("ID Ивана:", ivan_id)

assert anna_id is not None
assert ivan_id is not None

(1, 'Анна', 'anna@example.com')
(2, 'Иван', 'ivan@example.com')
ID Анны: 1
ID Ивана: 2


## Ячейка 8. Добавляем заказы, связанные с пользователями

У Анны будет два заказа, у Ивана один заказ.

In [8]:
orders = [
    (anna_id, "Ноутбук", 70000),
    (anna_id, "Мышь", 1500),
    (ivan_id, "Книга Python", 2500)
]

cursor.executemany(
    "INSERT INTO orders (user_id, title, price) VALUES (?, ?, ?)",
    orders
)

connection.commit()

print("Заказы добавлены")

Заказы добавлены


## Ячейка 9. Проверяем связанные данные через SELECT

Пока без JOIN просто смотрим обе таблицы.

In [9]:
print("Пользователи:")
cursor.execute("SELECT * FROM users")
users_rows = cursor.fetchall()
for row in users_rows:
    print(row)

print("\nЗаказы:")
cursor.execute("SELECT * FROM orders")
orders_rows = cursor.fetchall()
for row in orders_rows:
    print(row)

assert len(users_rows) == 2
assert len(orders_rows) == 3

Пользователи:
(1, 'Анна', 'anna@example.com')
(2, 'Иван', 'ivan@example.com')

Заказы:
(1, 1, 'Ноутбук', 70000)
(2, 1, 'Мышь', 1500)
(3, 2, 'Книга Python', 2500)


## Ячейка 10. Простой запрос: заказы конкретного пользователя

Получим все заказы Анны по её `user_id`.

In [10]:
cursor.execute(
    "SELECT * FROM orders WHERE user_id = ?",
    (anna_id,)
)

anna_orders = cursor.fetchall()

print("Заказы Анны:")
for order in anna_orders:
    print(order)

assert len(anna_orders) == 2

connection.close()
print("Соединение закрыто")

Заказы Анны:
(1, 1, 'Ноутбук', 70000)
(2, 1, 'Мышь', 1500)
Соединение закрыто


# Итог занятия 5

Сегодня вы научились:

- создавать связанные таблицы;
- понимать связь один ко многим;
- использовать `FOREIGN KEY`;
- включать `PRAGMA foreign_keys = ON`;
- добавлять данные в связанные таблицы;
- получать записи, связанные с конкретным id.

## Что коммитить в GitHub

- solved-ноутбук;
- TODO-ноутбук;
- SQL-код связанных таблиц;
- свою схему связи для диплома.

```bash
git add .
git commit -m "feat: add foreign keys and table relations"
git push -u origin feature/lesson-05-relations
```